# MedQuAD QLoRA fine-tuning — Qwen3-4B

Academic experiment using the conservative MedQuAD split. This notebook trains a LoRA adapter, not the full model. It is not intended for clinical deployment.

Before running: select **Runtime → Change runtime type → GPU** and upload the `dataset` directory to Google Drive.

In [ ]:
!nvidia-smi


## Install dependencies
If Colab asks for a runtime restart after installation, restart it and continue from the next cell.

In [ ]:
%%capture
!pip install -U unsloth unsloth_zoo datasets accelerate peft trl bitsandbytes


## Mount Drive and configure the run
Use `smoke` first. Change to `full` only after the smoke test finishes successfully.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/medquad_qwen3_4b')
DATA_DIR = PROJECT_DIR / 'dataset'
OUTPUT_DIR = PROJECT_DIR / 'outputs' / 'qwen3_4b_medquad_lora'

RUN_MODE = 'smoke'  # 'smoke' or 'full'
MODEL_NAME = 'unsloth/Qwen3-4B-unsloth-bnb-4bit'
MAX_SEQ_LENGTH = 2048
SEED = 3407

for filename in ('train.jsonl', 'valid.jsonl', 'test.jsonl'):
    path = DATA_DIR / filename
    assert path.exists(), f'Missing {path}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Data:', DATA_DIR)
print('Output:', OUTPUT_DIR)


## Load the leakage-safe dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    'json',
    data_files={
        'train': str(DATA_DIR / 'train.jsonl'),
        'validation': str(DATA_DIR / 'valid.jsonl'),
        'test': str(DATA_DIR / 'test.jsonl'),
    },
)
print(dataset)
print(dataset['train'][0]['messages'])


## Load Qwen3-4B in 4-bit and run a baseline answer

In [ ]:
import torch
from unsloth import FastLanguageModel, is_bfloat16_supported
from transformers import TextStreamer

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
    full_finetuning=False,
)

def generate_answer(question, max_new_tokens=256):
    messages = [
        dataset['test'][0]['messages'][0],
        {'role': 'user', 'content': question},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer(text, return_tensors='pt').to('cuda')
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        top_p=0.8,
        top_k=20,
        do_sample=True,
        repetition_penalty=1.05,
    )
    return tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True,
    ).strip()

baseline_question = dataset['test'][0]['messages'][1]['content']
print('Question:', baseline_question)
print('Base model:', generate_answer(baseline_question))


## Add LoRA adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
)


## Apply Qwen's non-thinking chat template
The dataset contains final answers, not reasoning traces, so thinking mode is disabled.

In [ ]:
def format_batch(examples):
    texts = [
        tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
            enable_thinking=False,
        )
        for messages in examples['messages']
    ]
    return {'text': texts}

formatted = dataset.map(format_batch, batched=True, num_proc=2)
if RUN_MODE == 'smoke':
    train_data = formatted['train'].shuffle(seed=SEED).select(range(1000))
    valid_data = formatted['validation'].select(range(200))
    test_data = formatted['test'].select(range(200))
else:
    train_data = formatted['train']
    valid_data = formatted['validation']
    test_data = formatted['test']

print('Train:', len(train_data), 'Validation:', len(valid_data), 'Test:', len(test_data))
print(train_data[0]['text'][:1000])


## Configure the SFT trainer
Prompt and system tokens are masked after trainer creation, so loss is calculated only on assistant responses.

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

training_args = SFTConfig(
    dataset_text_field='text',
    max_length=MAX_SEQ_LENGTH,
    packing=False,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1.0,
    max_steps=100 if RUN_MODE == 'smoke' else -1,
    learning_rate=1e-4,
    warmup_ratio=0.03,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    optim='adamw_8bit',
    weight_decay=0.01,
    lr_scheduler_type='linear',
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=50 if RUN_MODE == 'smoke' else 200,
    save_strategy='steps',
    save_steps=50 if RUN_MODE == 'smoke' else 200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    seed=SEED,
    output_dir=str(OUTPUT_DIR / 'checkpoints'),
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_data,
    eval_dataset=valid_data,
    args=training_args,
)
trainer = train_on_responses_only(trainer)
print('Trainer ready')


## Verify assistant-only masking
The decoded labels should display only the assistant answer; prompt tokens should appear as padding.

In [ ]:
sample = trainer.train_dataset[0]
visible_labels = [
    tokenizer.pad_token_id if token == -100 else token
    for token in sample['labels']
]
print(tokenizer.decode(visible_labels, skip_special_tokens=False)[:2000])


## Train
To resume after a disconnection, use `trainer.train(resume_from_checkpoint=True)` if a checkpoint exists.

In [ ]:
gpu = torch.cuda.get_device_properties(0)
print(f'GPU: {gpu.name}; memory: {gpu.total_memory / 1024**3:.1f} GB')
train_result = trainer.train()
print(train_result.metrics)


## Evaluate validation and held-out test loss

In [ ]:
validation_metrics = trainer.evaluate(eval_dataset=valid_data, metric_key_prefix='validation')
test_metrics = trainer.evaluate(eval_dataset=test_data, metric_key_prefix='test')
print(validation_metrics)
print(test_metrics)


## Compare the fine-tuned answer

In [ ]:
FastLanguageModel.for_inference(model)
print('Question:', baseline_question)
print('Fine-tuned model:', generate_answer(baseline_question))


## Save the LoRA adapter and tokenizer to Drive

In [ ]:
FINAL_ADAPTER_DIR = OUTPUT_DIR / 'final_adapter'
model.save_pretrained(str(FINAL_ADAPTER_DIR))
tokenizer.save_pretrained(str(FINAL_ADAPTER_DIR))
print('Saved:', FINAL_ADAPTER_DIR)


## Optional: export GGUF
This can take additional time and storage. Keep it disabled until adapter training and evaluation are complete.

In [ ]:
EXPORT_GGUF = False
if EXPORT_GGUF:
    model.save_pretrained_gguf(
        str(OUTPUT_DIR / 'gguf'),
        tokenizer,
        quantization_method='q4_k_m',
    )
